# Transformer Anh–Việt trên máy cá nhân

Mở notebook bằng kernel của môi trường đã cài PyTorch CUDA. Mô hình tự cài trong `scripts/transformer.py` dùng positional encoding hình sin, ba tầng encoder và decoder, bốn head, chiều 256 và FFN 1.024. Mỗi vị trí đích chỉ nhìn thấy tiền tố và các token nguồn khác PAD. Huấn luyện bằng teacher forcing, bỏ PAD khi tính cross-entropy. [Vaswani và cộng sự, 2017](https://arxiv.org/abs/1706.03762)

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/corpus.py").exists())
sys.path.insert(0, str(root))
import torch
from scripts.check_transformer import check
from scripts.train_transformer import fit, export, load_best
from scripts.transformer import translate, BOS, EOS
from scripts.download_data import download
download()
print(check())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Huấn luyện và xuất kết quả

Chọn một tokenizer hoặc lần lượt chạy cả ba. Mọi cấu hình dùng toàn bộ cùng tập train, seed 42, AdamW 3e-4, batch 32, cùng ngân sách một epoch cho baseline. Khi tăng ngân sách, dùng thư mục mới và cùng số epoch cho cả ba. Chọn checkpoint theo loss trên tst2012. Beam rộng 4 dùng tổng log-prob chia độ dài, tính cả EOS. Giới hạn 128 token sinh. Notebook 05 đọc các tệp test sau khi xuất.

In [ ]:
RUN_TRAIN = False
RUN_EXPORT = False
output = root / "artifacts"
for kind in ("bpe", "unigram", "word"):
    folder = output / "transformer" / kind
    if RUN_TRAIN:
        fit(kind, folder, device, epochs=1, batch=32)
    if RUN_EXPORT:
        export(folder, device, kind, output / "predictions")

## Thử dịch từ checkpoint đã lưu

In [ ]:
kind = "bpe"
folder = output / "transformer" / kind
if (folder / "best.pt").exists():
    model, (en, vi), config = load_best(folder, device)
    sentence = "Thank you very much ."
    source = torch.tensor([[BOS, *en.encode(sentence)[:config.limit-2], EOS]], device=device)
    for width in (1, 4):
        print(width, vi.decode(translate(model, source, width, config.limit)))